# Phase 4: Modelling (Part 2 - Baseline Support Vector Machine)
**Client:** Crédit Nationale Azur

**Objective:** Build an initial baseline Support Vector Machine (SVM) using all available features to establish a comparative benchmark.

In this notebook, we recreate our strict data splitting and encoding pipeline. Crucially, because SVMs are distance-based, we must also standardise all continuous features before training the model.

In [13]:
# Required Base Imports
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.svm import SVC

## 1. Data Preparation Pipeline
We execute the required sequence: loading the clean data, mapping the target variable, and performing our stratified 80/20 train/test split to prevent leakage.

In [14]:
# Load and map
df = joblib.load('../data/cleaned_data.pkl')
df['personal_loan'] = df['personal_loan'].map({'yes': 1, 'no': 0})
X = df.drop(['personal_loan', 'customer_id'], axis=1)
y = df['personal_loan']

# Split and copy
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train = X_train.copy()
X_test = X_test.copy()

## 2. One-Hot Encoding
We systematically encode the categorical text variables into integer dummies.

In [15]:
categorical_cols = ['education_level', 'credit_card_acct', 'online_acct']

for col in categorical_cols:
    dummies_train = pd.get_dummies(X_train[col], prefix=col, dtype=int)
    X_train.drop([col], axis=1, inplace=True)
    X_train = pd.concat([X_train, dummies_train], axis=1)
    
    dummies_test = pd.get_dummies(X_test[col], prefix=col, dtype=int)
    X_test.drop([col], axis=1, inplace=True)
    X_test = pd.concat([X_test, dummies_test], axis=1)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

## 3. Standardisation (All Continuous Features)
We apply `StandardScaler` to all continuous features across the dataset, ensuring the SVM's spatial calculations are not distorted by the raw numerical ranges.

In [16]:
# Define all continuous features
continuous_features = [
    'age', 'yrs_experience', 'family_size', 'income', 
    'mortgage_amt', 'credit_card_spend'
]

# Scale all continuous columns strictly after the split
for col in continuous_features:
    scaler = StandardScaler()
    scaler.fit(X_train[col].values.reshape(-1, 1))
    X_train[col] = scaler.transform(X_train[col].values.reshape(-1, 1)).flatten()
    X_test[col] = scaler.transform(X_test[col].values.reshape(-1, 1)).flatten()
    
print("Standardisation of all continuous features complete.")

Standardisation of all continuous features complete.


## 4. Train Baseline Model and Evaluate
We instantiate our Baseline SVM, train it on the scaled all-features dataset, and output the required performance metrics.

In [17]:
# Instantiate and fit the baseline SVM model
svm_model_all = SVC(random_state=42)
svm_model_all.fit(X_train, y_train)

# Generate predictions on the test set
y_pred_all = svm_model_all.predict(X_test)

# Flatten the confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_all).ravel()

# Calculate specific metrics
precision = precision_score(y_test, y_pred_all, zero_division=0)
recall = recall_score(y_test, y_pred_all)
f1 = f1_score(y_test, y_pred_all)

print("Baseline SVM Evaluation")
print("*" * 38)
print(f"True Negatives (TN):  {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP):  {tp}")
print("*" * 38)
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Baseline SVM Evaluation
**************************************
True Negatives (TN):  986
False Positives (FP): 34
False Negatives (FN): 55
True Positives (TP):  125
**************************************
Precision: 0.7862
Recall:    0.6944
F1-Score:  0.7375
